# HS10 Data Center Relevance Classifier

This notebook demonstrates how to classify HS10 commodity codes by their relevance to data center construction.

## Setup

1. Place `hs10_datacenter_classifier.py` in the same directory as this notebook (or your working directory)
2. Place your HS10 data file (`unique_hs10_commodities.csv`) in the same directory
3. Run the cells below

In [ ]:
# Import required libraries
import pandas as pd
import os

# Check that files exist
print("Files in current directory:")
for f in os.listdir('.'):
    if f.endswith('.py') or f.endswith('.csv'):
        print(f"  {f}")

## Option 1: Run the Script Directly

The simplest approach - just run the script with `%run`:

In [ ]:
# This will process the default input file and create the output
%run hs10_datacenter_classifier.py

## Option 2: Import and Use Functions

For more control, import the functions directly:

In [ ]:
# Import the classifier functions
from hs10_datacenter_classifier import (
    classify_hs10_file,
    classify_single_code,
    print_summary
)

In [ ]:
# Classify the full file
# Change these paths as needed
INPUT_FILE = "unique_hs10_commodities.csv"
OUTPUT_FILE = "hs10_datacenter_classification.csv"

results_df = classify_hs10_file(INPUT_FILE, OUTPUT_FILE)

In [ ]:
# Print summary statistics
print_summary(results_df)

## Exploring the Results

In [ ]:
# View the first few rows
results_df.head(10)

In [ ]:
# Filter to only HIGH relevance codes
high_relevance = results_df[results_df['Relevance'] == 'High']
print(f"Total HIGH relevance codes: {len(high_relevance)}")
high_relevance.head(20)

In [ ]:
# Filter to only MEDIUM relevance codes
medium_relevance = results_df[results_df['Relevance'] == 'Medium']
print(f"Total MEDIUM relevance codes: {len(medium_relevance)}")
medium_relevance.head(20)

In [ ]:
# View breakdown by materials category for HIGH codes
high_relevance['Materials_List_Mapping'].value_counts()

In [ ]:
# View which HS chapters have the most HIGH relevance codes
high_relevance['Chapter'] = high_relevance['HS10_Code'].astype(str).str[:2]
print("HIGH relevance codes by HS Chapter:")
high_relevance['Chapter'].value_counts().head(15)

## Classify Individual Codes

You can also classify individual descriptions without a file:

In [ ]:
# Test individual descriptions
test_descriptions = [
    "AUTOMATIC DATA PROCESSING MACHINES",
    "DIESEL GENERATORS FOR POWER GENERATION",
    "COPPER WIRE INSULATED FOR ELECTRICAL USE",
    "FRESH APPLES",
    "AIR CONDITIONING MACHINES",
    "OPTICAL FIBER CABLES",
    "LITHIUM-ION STORAGE BATTERIES"
]

for desc in test_descriptions:
    relevance, mapping = classify_single_code(desc)
    print(f"{relevance:6} | {mapping:50} | {desc}")

## Export Filtered Results

In [ ]:
# Export only HIGH and MEDIUM relevance codes (for trade analysis)
relevant_codes = results_df[results_df['Relevance'].isin(['High', 'Medium'])]
relevant_codes.to_csv('hs10_datacenter_relevant_only.csv', index=False)
print(f"Exported {len(relevant_codes)} relevant codes to 'hs10_datacenter_relevant_only.csv'")

In [ ]:
# Export only HIGH relevance codes
high_relevance.to_csv('hs10_datacenter_high_only.csv', index=False)
print(f"Exported {len(high_relevance)} HIGH relevance codes to 'hs10_datacenter_high_only.csv'")

## Customizing the Classifier

To add or modify classification rules, edit the `CLASSIFICATION_RULES` list in `hs10_datacenter_classifier.py`.

Each rule is a tuple: `(keyword, relevance, material_item, category)`

- `keyword`: The text to search for (case-insensitive, word boundaries)
- `relevance`: 'High', 'Medium', or 'Low'
- `material_item`: What data center material it maps to
- `category`: The broader category (e.g., 'Compute Components', 'Cooling Systems')

Rules are checked in order, so put more specific rules before general ones.